# SkillCorner Tracking Insights

In-possession shape, out-of-possession shape, and compactness using **mplsoccer** SkillCorner pitch coordinates.

| Pitch coord system | x | y |
|---|---|---|
| SkillCorner | −52.5 → +52.5 m | −34 → +34 m |

All shapes are normalised so the **team always attacks right** (positive x).

In [ ]:
# Run once to install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'mplsoccer==1.2.4', 'scipy', '-q'],
               capture_output=True)

In [ ]:
import json
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from mplsoccer import Pitch
from scipy.spatial import ConvexHull

plt.rcParams['figure.dpi'] = 130

## 1. Load data & assign teams

In [ ]:
TRACKING_FILE = "/Users/user/Downloads/2068343_tracking_extrapolated.jsonl"

print("Loading tracking data…")
frames = []
with open(TRACKING_FILE) as f:
    for line in f:
        frames.append(json.loads(line))
print(f"Loaded {len(frames):,} frames")

# Primary: team assignment from possession labels
player_team = {}
for fr in frames:
    pid, grp = fr['possession']['player_id'], fr['possession']['group']
    if pid and grp:
        player_team[pid] = grp

# Fallback: avg x in period 1 (home GK at negative x, away GK at positive x)
p1_x = defaultdict(list)
for fr in frames:
    if fr['period'] == 1:
        for p in fr['player_data']:
            if p['is_detected']:
                p1_x[p['player_id']].append(p['x'])
for pid, xs in p1_x.items():
    if pid not in player_team and len(xs) > 200:
        player_team[pid] = 'home team' if np.mean(xs) < 0 else 'away team'

home_ids = {pid for pid, t in player_team.items() if t == 'home team'}
away_ids = {pid for pid, t in player_team.items() if t == 'away team'}

avg_x   = {pid: np.mean(xs) for pid, xs in p1_x.items() if len(xs) > 500}
HOME_GK = min(home_ids & avg_x.keys(), key=lambda p: avg_x[p])
AWAY_GK = max(away_ids & avg_x.keys(), key=lambda p: avg_x[p])

print(f"Home: {len(home_ids)} players  (GK {HOME_GK})")
print(f"Away: {len(away_ids)} players  (GK {AWAY_GK})")

In [ ]:
print("Building DataFrame…")
rows = []
for fr in frames:
    if fr['period'] is None:
        continue
    poss_team = fr['possession']['group']
    period    = fr['period']
    for p in fr['player_data']:
        if not p['is_detected']:
            continue
        pid  = p['player_id']
        team = player_team.get(pid)
        if not team:
            continue
        rows.append((fr['frame'], period, poss_team, pid, team, p['x'], p['y']))

df = pd.DataFrame(rows, columns=['frame','period','poss_team','player_id','team','x','y'])
print(f"{len(df):,} rows · {df['player_id'].nunique()} players · "
      f"{df['poss_team'].notna().sum():,} rows with a possession label")
df.head(3)

## 2. Helpers

In [ ]:
HOME_COLOR       = '#1a78cf'
AWAY_COLOR       = '#e63946'
HOME_COLOR_LIGHT = '#7ab8e8'
AWAY_COLOR_LIGHT = '#f5959c'

PITCH_KW = dict(
    pitch_type='skillcorner',
    pitch_length=105,
    pitch_width=68,
    pitch_color='#1a2332',
    line_color='#4a5568',
    goal_type='box',
    linewidth=1.4,
)


def compute_shape(df, team, poss_state, gk_id):
    """Median outfield player positions normalised so team attacks right (+x)."""
    sub = df[(df['team'] == team) & (df['player_id'] != gk_id)].copy()

    if poss_state == 'in':
        sub = sub[sub['poss_team'] == team]
    else:
        opp = 'away team' if team == 'home team' else 'home team'
        sub = sub[sub['poss_team'] == opp]

    # Home attacks right in P1 (flip P2); Away attacks left in P1 (flip P1)
    flip      = (sub['period'] == 2) if team == 'home team' else (sub['period'] == 1)
    sub['xn'] = np.where(flip, -sub['x'],  sub['x'])
    sub['yn'] = np.where(flip, -sub['y'],  sub['y'])

    return sub.groupby('player_id')[['xn', 'yn']].median()


def hull_metrics(pts):
    """Return (area m², width m, depth m) for a point cloud."""
    if len(pts) < 3:
        return 0.0, 0.0, 0.0
    try:
        area = float(ConvexHull(pts).volume)  # scipy: 'volume' = area for 2-D hull
    except Exception:
        area = 0.0
    return area, float(pts[:, 1].max() - pts[:, 1].min()), float(pts[:, 0].max() - pts[:, 0].min())


def draw_hull(ax, pts, color, alpha_fill=0.18, alpha_edge=0.85, lw=2.2):
    """Draw filled convex hull."""
    if len(pts) < 3:
        return
    try:
        hull = ConvexHull(pts)
    except Exception:
        return
    verts  = pts[hull.vertices]
    closed = np.vstack([verts, verts[0]])
    ax.fill(verts[:, 0], verts[:, 1], color=color, alpha=alpha_fill, zorder=3)
    ax.plot(closed[:, 0], closed[:, 1], color=color, lw=lw, alpha=alpha_edge, zorder=4)


def metric_label(ax, text, color, y):
    ax.text(0, y, text, ha='center', va='bottom', color=color, fontsize=9.5,
            bbox=dict(boxstyle='round,pad=0.4', fc='#0d1117', alpha=0.75, ec=color, lw=0.8),
            zorder=10)


def attack_arrow(ax):
    ax.annotate('', xy=(49, 0), xytext=(39, 0),
                arrowprops=dict(arrowstyle='->', color='white', lw=1.5))


print("Helpers ready.")

## 3. In-Possession Shape

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('In-Possession Shape  (attacks  →)', color='white', fontsize=16, y=1.02)

for ax, (team, gk, color) in zip(axes, [
    ('home team', HOME_GK, HOME_COLOR),
    ('away team', AWAY_GK, AWAY_COLOR),
]):
    pitch = Pitch(**PITCH_KW)
    pitch.draw(ax=ax)

    shape = compute_shape(df, team, 'in', gk)
    pts   = shape[['xn', 'yn']].values

    draw_hull(ax, pts, color)
    pitch.scatter(pts[:, 0], pts[:, 1], ax=ax, s=260,
                  color=color, edgecolors='white', linewidths=1.5, zorder=5)

    for pid, row in shape.iterrows():
        ax.text(row['xn'], row['yn'] - 3.2, str(pid)[-5:],
                ha='center', va='top', color='white', fontsize=7.5,
                fontweight='bold', zorder=6)

    # Centroid marker
    cx, cy = float(shape['xn'].mean()), float(shape['yn'].mean())
    ax.scatter(cx, cy, s=140, marker='+', color='yellow', lw=2.5, zorder=7)

    area, width, depth = hull_metrics(pts)
    label = 'Home' if team == 'home team' else 'Away'
    ax.set_title(f'{label}  —  In Possession', color='white', fontsize=13, pad=8)
    metric_label(ax, f'Width {width:.0f} m  ·  Depth {depth:.0f} m  ·  Area {area:.0f} m²', color, -36.5)
    attack_arrow(ax)

plt.tight_layout()
plt.savefig('in_possession_shape.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 4. Out-of-Possession Shape

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Out-of-Possession Shape  (attacks  →)', color='white', fontsize=16, y=1.02)

for ax, (team, gk, color) in zip(axes, [
    ('home team', HOME_GK, HOME_COLOR),
    ('away team', AWAY_GK, AWAY_COLOR),
]):
    pitch = Pitch(**PITCH_KW)
    pitch.draw(ax=ax)

    shape = compute_shape(df, team, 'out', gk)
    pts   = shape[['xn', 'yn']].values

    draw_hull(ax, pts, color)
    pitch.scatter(pts[:, 0], pts[:, 1], ax=ax, s=260,
                  color=color, edgecolors='white', linewidths=1.5, zorder=5)

    for pid, row in shape.iterrows():
        ax.text(row['xn'], row['yn'] - 3.2, str(pid)[-5:],
                ha='center', va='top', color='white', fontsize=7.5,
                fontweight='bold', zorder=6)

    cx, cy = float(shape['xn'].mean()), float(shape['yn'].mean())
    ax.scatter(cx, cy, s=140, marker='+', color='yellow', lw=2.5, zorder=7)

    area, width, depth = hull_metrics(pts)
    label = 'Home' if team == 'home team' else 'Away'
    ax.set_title(f'{label}  —  Out of Possession', color='white', fontsize=13, pad=8)
    metric_label(ax, f'Width {width:.0f} m  ·  Depth {depth:.0f} m  ·  Area {area:.0f} m²', color, -36.5)
    attack_arrow(ax)

plt.tight_layout()
plt.savefig('out_of_possession_shape.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 5. Compactness Overlay

Both shapes (in-possession **○** and out-of-possession **◇**) overlaid on the same pitch.  
Arrows show how each player shifts between the two phases.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Compactness: In-Possession vs Out-of-Possession  (attacks  →)',
             color='white', fontsize=16, y=1.02)

for ax, (team, gk, c_in, c_out) in zip(axes, [
    ('home team', HOME_GK, HOME_COLOR_LIGHT, HOME_COLOR),
    ('away team', AWAY_GK, AWAY_COLOR_LIGHT, AWAY_COLOR),
]):
    pitch = Pitch(**PITCH_KW)
    pitch.draw(ax=ax)

    shape_in  = compute_shape(df, team, 'in',  gk)
    shape_out = compute_shape(df, team, 'out', gk)
    pts_in    = shape_in [['xn', 'yn']].values
    pts_out   = shape_out[['xn', 'yn']].values

    # Hulls: out underneath, in on top
    draw_hull(ax, pts_out, c_out, alpha_fill=0.25, alpha_edge=0.9,  lw=2.5)
    draw_hull(ax, pts_in,  c_in,  alpha_fill=0.25, alpha_edge=0.9,  lw=2.5)

    # Player dots
    pitch.scatter(pts_out[:, 0], pts_out[:, 1], ax=ax, s=210, marker='D',
                  color=c_out, edgecolors='white', linewidths=1.2, zorder=5)
    pitch.scatter(pts_in [:, 0], pts_in [:, 1], ax=ax, s=210, marker='o',
                  color=c_in,  edgecolors='white', linewidths=1.2, zorder=6)

    # Shift arrows: in-possession → out-of-possession
    for pid in set(shape_in.index) & set(shape_out.index):
        xi, yi = float(shape_in.loc[pid,  'xn']), float(shape_in.loc[pid,  'yn'])
        xo, yo = float(shape_out.loc[pid, 'xn']), float(shape_out.loc[pid, 'yn'])
        ax.annotate('', xy=(xo, yo), xytext=(xi, yi),
                    arrowprops=dict(arrowstyle='->', color='white', lw=0.9, alpha=0.45))

    # Metrics annotations
    for pts, c, lbl, yp in [(pts_in, c_in, 'In-poss', -32.8), (pts_out, c_out, 'Out-poss', -36.3)]:
        area, width, depth = hull_metrics(pts)
        metric_label(ax, f'{lbl}: W={width:.0f}m  D={depth:.0f}m  A={area:.0f}m²', c, yp)

    label = 'Home' if team == 'home team' else 'Away'
    ax.set_title(label, color='white', fontsize=13, pad=8)

    handles = [
        mpatches.Patch(fc=c_in,  ec='white', label='In possession  (○)'),
        mpatches.Patch(fc=c_out, ec='white', label='Out of possession  (◇)'),
    ]
    ax.legend(handles=handles, facecolor='#1c2128', labelcolor='white',
              loc='upper left', fontsize=9, framealpha=0.85)
    attack_arrow(ax)

plt.tight_layout()
plt.savefig('compactness_overlay.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 6. Match Summary Stats

In [ ]:
poss_df   = df[df['poss_team'].notna()]
home_poss = (poss_df['poss_team'] == 'home team').sum()
away_poss = (poss_df['poss_team'] == 'away team').sum()
total     = home_poss + away_poss

# Compactness summary
for team, gk, label in [('home team', HOME_GK, 'Home'), ('away team', AWAY_GK, 'Away')]:
    s_in  = compute_shape(df, team, 'in',  gk)
    s_out = compute_shape(df, team, 'out', gk)
    ai, wi, di = hull_metrics(s_in [['xn','yn']].values)
    ao, wo, do = hull_metrics(s_out[['xn','yn']].values)
    print(f"{label}:")
    print(f"  In-possession  → W={wi:.0f}m  D={di:.0f}m  Area={ai:.0f}m²")
    print(f"  Out-poss       → W={wo:.0f}m  D={do:.0f}m  Area={ao:.0f}m²")
    print(f"  Compactness change: area {'▲' if ao < ai else '▼'} {abs(ao-ai):.0f}m² when defending")
    print()

print(f"Possession — Home {home_poss/total*100:.1f}%  |  Away {away_poss/total*100:.1f}%")